In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

men_regular = pd.read_csv(
    RAW / "MRegularSeasonDetailedResults.csv"
)

women_regular = pd.read_csv(
    RAW / "WRegularSeasonDetailedResults.csv"
)

men_original = pd.read_csv(
    PROCESSED / "men_processed.csv"
)

women_original = pd.read_csv(
    PROCESSED / "women_processed.csv"
)

print("Men regular:", men_regular.shape)
print("Women regular:", women_regular.shape)
print("Men matchups:", men_original.shape)
print("Women matchups:", women_original.shape)

Men regular: (124529, 34)
Women regular: (87187, 34)
Men matchups: (1449, 8)
Women matchups: (961, 8)


In [2]:
def calculate_team_netrating(regular):

    df = regular.copy()

    # Estimate possessions for both teams
    winner_poss = (
        df["WFGA"] - df["WOR"]
        + df["WTO"] + 0.475 * df["WFTA"]
    )

    loser_poss = (
        df["LFGA"] - df["LOR"]
        + df["LTO"] + 0.475 * df["LFTA"]
    )

    # Both teams share approximately the same possessions
    df["GamePoss"] = (winner_poss + loser_poss) / 2

    assert (df["GamePoss"] > 0).all()

    # Winner perspective
    winners = pd.DataFrame({
        "Season": df["Season"],
        "TeamID": df["WTeamID"],
        "PointsFor": df["WScore"],
        "PointsAgainst": df["LScore"],
        "Possessions": df["GamePoss"]
    })

    # Loser perspective
    losers = pd.DataFrame({
        "Season": df["Season"],
        "TeamID": df["LTeamID"],
        "PointsFor": df["LScore"],
        "PointsAgainst": df["WScore"],
        "Possessions": df["GamePoss"]
    })

    team_games = pd.concat(
        [winners, losers],
        ignore_index=True
    )

    # Aggregate by team and season
    team_stats = (
        team_games.groupby(
            ["Season", "TeamID"],
            as_index=False
        ).agg(
            GamesPlayed=("Possessions", "size"),
            PointsFor=("PointsFor", "sum"),
            PointsAgainst=("PointsAgainst", "sum"),
            TotalPossessions=("Possessions", "sum")
        )
    )

    # Efficiency per 100 possessions
    team_stats["OffRating"] = (
        100 * team_stats["PointsFor"]
        / team_stats["TotalPossessions"]
    )

    team_stats["DefRating"] = (
        100 * team_stats["PointsAgainst"]
        / team_stats["TotalPossessions"]
    )

    team_stats["NetRating"] = (
        team_stats["OffRating"]
        - team_stats["DefRating"]
    )

    assert np.isfinite(
        team_stats[
            ["OffRating", "DefRating", "NetRating"]
        ].to_numpy()
    ).all()

    assert not team_stats.duplicated(
        ["Season", "TeamID"]
    ).any()

    return team_stats

In [3]:
men_team_stats = calculate_team_netrating(men_regular)
women_team_stats = calculate_team_netrating(women_regular)

men_team_stats.to_csv(
    PROCESSED / "men_netrating_team_season.csv",
    index=False
)

women_team_stats.to_csv(
    PROCESSED / "women_netrating_team_season.csv",
    index=False
)

print("Men team-season:", men_team_stats.shape)
print("Women team-season:", women_team_stats.shape)

display(men_team_stats.head())
display(women_team_stats.head())

Men team-season: (8346, 9)
Women team-season: (5965, 9)


,Season,TeamID,GamesPlayed,PointsFor,PointsAgainst,TotalPossessions,OffRating,DefRating,NetRating
0,2003,1102,28,1603,1596,1541.2750,104.004801,103.550632,0.454169
1,2003,1103,27,2127,2110,1914.3000,111.111111,110.223058,0.888053
2,2003,1104,28,1940,1820,1868.1750,103.844661,97.421280,6.423381
3,2003,1105,26,1866,1993,1993.6875,93.595411,99.965516,-6.370106
4,2003,1106,28,1781,1785,1896.0500,93.932122,94.143087,-0.210965


,Season,TeamID,GamesPlayed,PointsFor,PointsAgainst,TotalPossessions,OffRating,DefRating,NetRating
0,2010,3102,28,1451,2010,1815.9125,79.904731,110.688153,-30.783422
1,2010,3103,30,1867,1789,2064.3375,90.440638,86.662186,3.778452
2,2010,3104,29,1860,1943,2170.9500,85.676777,89.499988,-3.823211
3,2010,3105,27,1641,1748,1998.3500,82.117747,87.472165,-5.354417
4,2010,3106,29,1599,1665,2012.3875,79.457858,82.737544,-3.279686


In [4]:
def add_netrating(original, team_stats):

    keys = ["Season", "Team1", "Team2"]

    assert original["Team1"].lt(original["Team2"]).all()
    assert not original.duplicated(keys).any()

    team1 = team_stats[
        ["Season", "TeamID", "NetRating"]
    ].rename(columns={
        "TeamID": "Team1",
        "NetRating": "Team1NetRating"
    })

    team2 = team_stats[
        ["Season", "TeamID", "NetRating"]
    ].rename(columns={
        "TeamID": "Team2",
        "NetRating": "Team2NetRating"
    })

    merged = original.merge(
        team1,
        on=["Season", "Team1"],
        how="left",
        validate="many_to_one",
        sort=False
    )

    merged = merged.merge(
        team2,
        on=["Season", "Team2"],
        how="left",
        validate="many_to_one",
        sort=False
    )

    assert len(merged) == len(original)
    assert merged[
        ["Team1NetRating", "Team2NetRating"]
    ].notna().all().all()

    merged["NetRatingDiff"] = (
        merged["Team1NetRating"]
        - merged["Team2NetRating"]
    )

    # Check that the original features and targets remain unchanged
    pd.testing.assert_frame_equal(
        merged[original.columns].reset_index(drop=True),
        original.reset_index(drop=True),
        check_exact=True
    )

    return merged

In [5]:
men_advanced = add_netrating(
    men_original,
    men_team_stats
)

women_advanced = add_netrating(
    women_original,
    women_team_stats
)

men_advanced.to_csv(
    PROCESSED / "men_netrating_merged.csv",
    index=False
)

women_advanced.to_csv(
    PROCESSED / "women_netrating_merged.csv",
    index=False
)

print("Men merged:", men_advanced.shape)
print("Women merged:", women_advanced.shape)

display(men_advanced.head())
display(women_advanced.head())

Men merged: (1449, 11)
Women merged: (961, 11)


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target,Team1NetRating,Team2NetRating,NetRatingDiff
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0,2.762884,-10.238970,13.001854
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1,19.429181,7.041519,12.387662
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1,9.734103,12.323115,-2.589012
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1,8.448032,21.495511,-13.047479
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1,6.664721,6.526979,0.137741


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target,Team1NetRating,Team2NetRating,NetRatingDiff
0,2010,3124,3201,-9,2.371212,-0.099432,0.053358,1,21.128971,17.877800,3.251171
1,2010,3173,3395,-1,-0.730769,0.074359,-0.005113,1,15.459090,16.350723,-0.891633
2,2010,3181,3214,-13,8.831250,0.210417,0.054759,1,22.578494,11.035998,11.542496
3,2010,3199,3256,-11,4.431183,0.091398,0.039516,1,19.433892,13.050986,6.382906
4,2010,3207,3265,-7,-0.606061,0.012121,-0.087561,1,13.774186,15.234414,-1.460228


In [6]:
for name, df in [
    ("MEN", men_advanced),
    ("WOMEN", women_advanced)
]:
    print(f"\n{name}")
    print("Missing values:", df.isna().sum().sum())

    print(
        "Duplicate matchups:",
        df.duplicated(
            ["Season", "Team1", "Team2"]
        ).sum()
    )

    print(
        "NetRatingDiff range:",
        df["NetRatingDiff"].min(),
        "to",
        df["NetRatingDiff"].max()
    )

    correlation = df[
        ["AvgScoringMarginDiff", "NetRatingDiff"]
    ].corr().iloc[0, 1]

    print(
        "Correlation with scoring margin:",
        round(correlation, 4)
    )


MEN
Missing values: 0
Duplicate matchups: 0
NetRatingDiff range: -33.66096656621433 to 34.55838234873926
Correlation with scoring margin: 0.9937

WOMEN
Missing values: 0
Duplicate matchups: 0
NetRatingDiff range: -43.62461690200236 to 58.956100879502344
Correlation with scoring margin: 0.9967


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

original_features = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

feature_sets = {
    "A_Original4": original_features,

    "B_ReplaceMargin": [
        "SeedDiff",
        "NetRatingDiff",
        "WinRateDiff",
        "EFGDifferentialDiff"
    ],

    "C_AddNetRating": original_features + [
        "NetRatingDiff"
    ]
}

men_params = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42
}

women_params = {
    "max_depth": 1,
    "learning_rate": 0.30,
    "n_estimators": 15,
    "subsample": 0.5,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42
}

validation_years = [2017, 2018, 2019, 2021]


def run_feature_experiments(data, params, dataset_name):

    results = []

    for experiment, features in feature_sets.items():

        for year in validation_years:

            train = data[data["Season"] < year]
            val = data[data["Season"] == year]

            model = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                **params
            )

            model.fit(
                train[features],
                train["Target"]
            )

            probabilities = model.predict_proba(
                val[features]
            )[:, 1]

            brier = brier_score_loss(
                val["Target"],
                probabilities
            )

            results.append({
                "Dataset": dataset_name,
                "Experiment": experiment,
                "ValidationSeason": year,
                "BrierScore": brier
            })

    return pd.DataFrame(results)


men_results = run_feature_experiments(
    men_advanced,
    men_params,
    "Men"
)

women_results = run_feature_experiments(
    women_advanced,
    women_params,
    "Women"
)

all_results = pd.concat(
    [men_results, women_results],
    ignore_index=True
)

summary = (
    all_results
    .groupby(["Dataset", "Experiment"])["BrierScore"]
    .mean()
    .reset_index(name="MeanCVBrier")
)

print("Per-season results:")
display(all_results)

print("\nMean CV Brier:")
display(summary)

Per-season results:


,Dataset,Experiment,ValidationSeason,BrierScore
0,Men,A_Original4,2017,0.164758
1,Men,A_Original4,2018,0.220298
2,Men,A_Original4,2019,0.162818
3,Men,A_Original4,2021,0.218945
4,Men,B_ReplaceMargin,2017,0.161280
5,Men,B_ReplaceMargin,2018,0.219731
6,Men,B_ReplaceMargin,2019,0.163811
7,Men,B_ReplaceMargin,2021,0.215367
8,Men,C_AddNetRating,2017,0.165530
9,Men,C_AddNetRating,2018,0.221414



Mean CV Brier:


,Dataset,Experiment,MeanCVBrier
0,Men,A_Original4,0.191705
1,Men,B_ReplaceMargin,0.190047
2,Men,C_AddNetRating,0.192242
3,Women,A_Original4,0.146503
4,Women,B_ReplaceMargin,0.148292
5,Women,C_AddNetRating,0.144865


In [8]:
reference_scores = {
    "Men": 0.191705,
    "Women": 0.146503
}

# Verify the original four-feature control
for dataset, expected in reference_scores.items():

    actual = summary.loc[
        (summary["Dataset"] == dataset)
        & (summary["Experiment"] == "A_Original4"),
        "MeanCVBrier"
    ].iloc[0]

    print(
        f"{dataset} original control: "
        f"{actual:.6f} | "
        f"Previously recorded: {expected:.6f}"
    )

    assert abs(actual - expected) < 0.000002, (
        f"{dataset} control differs from earlier results. "
        "Check data, parameters and environment before "
        "interpreting Experiments B and C."
    )

print("\nAll original controls verified.")

Men original control: 0.191705 | Previously recorded: 0.191705
Women original control: 0.146503 | Previously recorded: 0.146503

All original controls verified.


In [9]:
from sklearn.linear_model import Ridge
from scipy.sparse import csr_matrix


def calculate_adjusted_strength(regular_results, alpha=20.0):

    all_ratings = []

    # Estimate ratings independently for each season
    for season, games in regular_results.groupby("Season"):

        games = games.reset_index(drop=True)

        # All teams appearing in this season
        team_ids = np.sort(
            np.unique(
                np.concatenate([
                    games["WTeamID"].to_numpy(),
                    games["LTeamID"].to_numpy()
                ])
            )
        )

        team_to_index = {
            team_id: i
            for i, team_id in enumerate(team_ids)
        }

        n_games = len(games)
        n_teams = len(team_ids)

        # Identify winning and losing teams
        winner_indices = (
            games["WTeamID"]
            .map(team_to_index)
            .to_numpy()
        )

        loser_indices = (
            games["LTeamID"]
            .map(team_to_index)
            .to_numpy()
        )

        # Location from winning team's perspective
        # H = winner at home, A = winner away, N = neutral
        location = games["WLoc"].map({
            "H": 1.0,
            "A": -1.0,
            "N": 0.0
        })

        assert location.notna().all(), (
            f"Unknown game location in season {season}"
        )

        location = location.to_numpy()

        # Build sparse regression design matrix:
        # +1 winning team
        # -1 losing team
        # +1/-1/0 for home advantage
        row_indices = np.concatenate([
            np.arange(n_games),
            np.arange(n_games),
            np.arange(n_games)
        ])

        column_indices = np.concatenate([
            winner_indices,
            loser_indices,
            np.full(n_games, n_teams)
        ])

        values = np.concatenate([
            np.ones(n_games),
            -np.ones(n_games),
            location
        ])

        X = csr_matrix(
            (values, (row_indices, column_indices)),
            shape=(n_games, n_teams + 1)
        )

        # Target: actual winning margin
        y = (
            games["WScore"] - games["LScore"]
        ).to_numpy(dtype=float)

        # Fit regularized linear regression
        model = Ridge(
            alpha=alpha,
            fit_intercept=False,
            solver="lsqr"
        )

        model.fit(X, y)

        # First n_teams coefficients are strength ratings
        team_strengths = model.coef_[:n_teams]

        # Last coefficient is home-court advantage
        home_advantage = model.coef_[n_teams]

        season_ratings = pd.DataFrame({
            "Season": season,
            "TeamID": team_ids,
            "AdjustedStrength": team_strengths,
            "HomeAdvantage": home_advantage
        })

        all_ratings.append(season_ratings)

    result = pd.concat(all_ratings, ignore_index=True)

    assert not result.duplicated(
        ["Season", "TeamID"]
    ).any()

    assert np.isfinite(
        result["AdjustedStrength"]
    ).all()

    return result

In [10]:
men_strength = calculate_adjusted_strength(
    men_regular,
    alpha=20.0
)

women_strength = calculate_adjusted_strength(
    women_regular,
    alpha=20.0
)

men_strength.to_csv(
    PROCESSED / "men_adjusted_strength_team_season.csv",
    index=False
)

women_strength.to_csv(
    PROCESSED / "women_adjusted_strength_team_season.csv",
    index=False
)

print("Men's adjusted strength:", men_strength.shape)
print("Women's adjusted strength:", women_strength.shape)

display(men_strength.head())
display(women_strength.head())

print(
    "\nMen's estimated 2025 home advantage:",
    men_strength.loc[
        men_strength["Season"] == 2025,
        "HomeAdvantage"
    ].iloc[0]
)

print(
    "Women's estimated 2025 home advantage:",
    women_strength.loc[
        women_strength["Season"] == 2025,
        "HomeAdvantage"
    ].iloc[0]
)

Men's adjusted strength: (8346, 4)
Women's adjusted strength: (5965, 4)


,Season,TeamID,AdjustedStrength,HomeAdvantage
0,2003,1102,0.413722,5.420873
1,2003,1103,-0.038397,5.420873
2,2003,1104,3.518437,5.420873
3,2003,1105,-4.302509,5.420873
4,2003,1106,-1.098150,5.420873


,Season,TeamID,AdjustedStrength,HomeAdvantage
0,2010,3102,-11.130089,4.614865
1,2010,3103,1.581360,4.614865
2,2010,3104,-0.155182,4.614865
3,2010,3105,-3.653953,4.614865
4,2010,3106,-2.812808,4.614865



Men's estimated 2025 home advantage: 4.785007556133275
Women's estimated 2025 home advantage: 4.386625349311266


In [11]:
def add_adjusted_strength(matchups, strength_table):

    original = matchups.copy()

    # Only use required columns
    ratings = strength_table[
        ["Season", "TeamID", "AdjustedStrength"]
    ].copy()

    assert not ratings.duplicated(
        ["Season", "TeamID"]
    ).any()

    team1 = ratings.rename(columns={
        "TeamID": "Team1",
        "AdjustedStrength": "Team1AdjustedStrength"
    })

    team2 = ratings.rename(columns={
        "TeamID": "Team2",
        "AdjustedStrength": "Team2AdjustedStrength"
    })

    merged = original.merge(
        team1,
        on=["Season", "Team1"],
        how="left",
        validate="many_to_one",
        sort=False
    )

    merged = merged.merge(
        team2,
        on=["Season", "Team2"],
        how="left",
        validate="many_to_one",
        sort=False
    )

    assert len(merged) == len(original)

    assert merged[
        ["Team1AdjustedStrength", "Team2AdjustedStrength"]
    ].notna().all().all()

    merged["AdjustedStrengthDiff"] = (
        merged["Team1AdjustedStrength"]
        - merged["Team2AdjustedStrength"]
    )

    # Confirm original columns and labels remain identical
    pd.testing.assert_frame_equal(
        merged[original.columns].reset_index(drop=True),
        original.reset_index(drop=True),
        check_exact=True
    )

    return merged


men_full = add_adjusted_strength(
    men_advanced,
    men_strength
)

women_full = add_adjusted_strength(
    women_advanced,
    women_strength
)

men_full.to_csv(
    PROCESSED / "men_advanced_merged.csv",
    index=False
)

women_full.to_csv(
    PROCESSED / "women_advanced_merged.csv",
    index=False
)

print("Men's advanced dataset:", men_full.shape)
print("Women's advanced dataset:", women_full.shape)

display(
    men_full[
        [
            "Season",
            "Team1",
            "Team2",
            "AvgScoringMarginDiff",
            "NetRatingDiff",
            "AdjustedStrengthDiff",
            "Target"
        ]
    ].head()
)

Men's advanced dataset: (1449, 14)
Women's advanced dataset: (961, 14)


,Season,Team1,Team2,AvgScoringMarginDiff,NetRatingDiff,AdjustedStrengthDiff,Target
0,2003,1411,1421,9.208046,13.001854,3.725507,0
1,2003,1112,1436,10.309113,12.387662,7.210406,1
2,2003,1113,1272,-1.896552,-2.589012,-0.399860,1
3,2003,1141,1166,-8.805643,-13.047479,-4.862680,1
4,2003,1143,1301,0.324138,0.137741,0.140835,1


In [12]:
# Original four features
original_4 = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

# Best men's Net Rating candidate
men_netrating_features = [
    "SeedDiff",
    "NetRatingDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

# Best women's Net Rating candidate
women_netrating_features = original_4 + [
    "NetRatingDiff"
]

men_feature_sets = {
    "A_Original4": original_4,

    "B_NetRating": men_netrating_features,

    "D_Original4_PlusStrength": original_4 + [
        "AdjustedStrengthDiff"
    ],

    "E_NetRating_PlusStrength": men_netrating_features + [
        "AdjustedStrengthDiff"
    ]
}

women_feature_sets = {
    "A_Original4": original_4,

    "C_Original4_PlusNetRating": women_netrating_features,

    "D_Original4_PlusStrength": original_4 + [
        "AdjustedStrengthDiff"
    ],

    "E_NetRating_PlusStrength": women_netrating_features + [
        "AdjustedStrengthDiff"
    ]
}


def evaluate_strength_features(
    data,
    params,
    feature_sets,
    dataset_name
):

    results = []

    for experiment, features in feature_sets.items():

        for year in validation_years:

            train = data[data["Season"] < year]
            valid = data[data["Season"] == year]

            model = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                **params
            )

            model.fit(
                train[features],
                train["Target"]
            )

            predictions = model.predict_proba(
                valid[features]
            )[:, 1]

            score = brier_score_loss(
                valid["Target"],
                predictions
            )

            results.append({
                "Dataset": dataset_name,
                "Experiment": experiment,
                "ValidationSeason": year,
                "BrierScore": score
            })

    return pd.DataFrame(results)


men_strength_results = evaluate_strength_features(
    men_full,
    men_params,
    men_feature_sets,
    "Men"
)

women_strength_results = evaluate_strength_features(
    women_full,
    women_params,
    women_feature_sets,
    "Women"
)

strength_results = pd.concat(
    [men_strength_results, women_strength_results],
    ignore_index=True
)

strength_summary = (
    strength_results
    .groupby(["Dataset", "Experiment"])["BrierScore"]
    .mean()
    .reset_index(name="MeanCVBrier")
)

print("Per-season validation results:")
display(strength_results)

print("\nMean CV Brier comparison:")
display(strength_summary.sort_values(
    ["Dataset", "MeanCVBrier"]
))

Per-season validation results:


,Dataset,Experiment,ValidationSeason,BrierScore
0,Men,A_Original4,2017,0.164758
1,Men,A_Original4,2018,0.220298
2,Men,A_Original4,2019,0.162818
3,Men,A_Original4,2021,0.218945
4,Men,B_NetRating,2017,0.161280
5,Men,B_NetRating,2018,0.219731
6,Men,B_NetRating,2019,0.163811
7,Men,B_NetRating,2021,0.215367
8,Men,D_Original4_PlusStrength,2017,0.163702
9,Men,D_Original4_PlusStrength,2018,0.213204



Mean CV Brier comparison:


,Dataset,Experiment,MeanCVBrier
3,Men,E_NetRating_PlusStrength,0.185850
2,Men,D_Original4_PlusStrength,0.186785
1,Men,B_NetRating,0.190047
0,Men,A_Original4,0.191705
5,Women,C_Original4_PlusNetRating,0.144865
4,Women,A_Original4,0.146503
6,Women,D_Original4_PlusStrength,0.148985
7,Women,E_NetRating_PlusStrength,0.149159


In [13]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

# ============================================================
# 1. FEATURE CONFIGURATIONS
# ============================================================

original_4 = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

men_new_features = [
    "SeedDiff",
    "NetRatingDiff",
    "WinRateDiff",
    "EFGDifferentialDiff",
    "AdjustedStrengthDiff"
]

women_new_features = original_4 + [
    "NetRatingDiff"
]


# ============================================================
# 2. XGBOOST HYPERPARAMETERS
# ============================================================

baseline_params = {
    "max_depth": 3,
    "learning_rate": 0.05,
    "n_estimators": 100,
    "subsample": 0.8,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}

men_tuned_params = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}

women_tuned_params = {
    "max_depth": 1,
    "learning_rate": 0.30,
    "n_estimators": 15,
    "subsample": 0.5,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}


# ============================================================
# 3. EVALUATION FUNCTION
# ============================================================

def evaluate_holdout(data, configurations, dataset_name):

    train = data[data["Season"] <= 2021].copy()
    test = data[data["Season"].between(2022, 2025)].copy()

    assert len(train) > 0
    assert len(test) > 0
    assert train["Season"].max() < test["Season"].min()

    results = []

    for experiment_name, (features, params) in configurations.items():

        model = XGBClassifier(**params)

        model.fit(
            train[features],
            train["Target"]
        )

        predictions = model.predict_proba(
            test[features]
        )[:, 1]

        overall_brier = brier_score_loss(
            test["Target"],
            predictions
        )

        print(
            f"{dataset_name} | {experiment_name} | "
            f"Overall Brier: {overall_brier:.6f}"
        )

        # Compute Brier separately for each holdout season
        for year in [2022, 2023, 2024, 2025]:

            mask = (test["Season"] == year).to_numpy()

            year_brier = brier_score_loss(
                test.loc[test["Season"] == year, "Target"],
                predictions[mask]
            )

            results.append({
                "Dataset": dataset_name,
                "Experiment": experiment_name,
                "Season": year,
                "BrierScore": year_brier,
                "OverallBrier": overall_brier
            })

    return pd.DataFrame(results)


# ============================================================
# 4. MEN'S HOLDOUT EXPERIMENTS
# ============================================================

men_configurations = {
    "A_OriginalBaseline": (
        original_4,
        baseline_params
    ),

    "B_TunedOriginal": (
        original_4,
        men_tuned_params
    ),

    "C_NewFeatures": (
        men_new_features,
        men_tuned_params
    )
}

men_holdout = evaluate_holdout(
    men_full,
    men_configurations,
    "Men"
)


# ============================================================
# 5. WOMEN'S HOLDOUT EXPERIMENTS
# ============================================================

women_configurations = {
    "A_OriginalBaseline": (
        original_4,
        baseline_params
    ),

    "B_TunedOriginal": (
        original_4,
        women_tuned_params
    ),

    "C_NewFeatures": (
        women_new_features,
        women_tuned_params
    )
}

women_holdout = evaluate_holdout(
    women_full,
    women_configurations,
    "Women"
)


# ============================================================
# 6. RESULTS SUMMARY
# ============================================================

holdout_results = pd.concat(
    [men_holdout, women_holdout],
    ignore_index=True
)

print("\nPER-SEASON HOLDOUT RESULTS")
display(
    holdout_results[
        ["Dataset", "Experiment", "Season", "BrierScore"]
    ]
)

holdout_summary = (
    holdout_results
    .groupby(["Dataset", "Experiment"], as_index=False)
    .agg(
        MeanHoldoutBrier=("OverallBrier", "first")
    )
)

print("\nOVERALL HOLDOUT COMPARISON")
display(
    holdout_summary.sort_values(
        ["Dataset", "MeanHoldoutBrier"]
    )
)


# ============================================================
# 7. VERIFY PREVIOUSLY RECORDED HOLDOUT SCORES
# ============================================================

reference_scores = {
    ("Men", "A_OriginalBaseline"): 0.2100603564,
    ("Men", "B_TunedOriginal"): 0.2128896757,
    ("Women", "A_OriginalBaseline"): 0.1507969948,
    ("Women", "B_TunedOriginal"): 0.1453367763
}

for (dataset, experiment), expected in reference_scores.items():

    actual = holdout_summary.loc[
        (holdout_summary["Dataset"] == dataset)
        & (holdout_summary["Experiment"] == experiment),
        "MeanHoldoutBrier"
    ].iloc[0]

    print(
        f"{dataset} | {experiment}: "
        f"actual={actual:.6f}, expected={expected:.6f}"
    )

    assert abs(actual - expected) < 0.000002, (
        f"Holdout mismatch for {dataset}, {experiment}"
    )

print("\nPreviously recorded holdout results reproduced.")

Men | A_OriginalBaseline | Overall Brier: 0.210060
Men | B_TunedOriginal | Overall Brier: 0.212890
Men | C_NewFeatures | Overall Brier: 0.215652
Women | A_OriginalBaseline | Overall Brier: 0.150797
Women | B_TunedOriginal | Overall Brier: 0.145337
Women | C_NewFeatures | Overall Brier: 0.148266

PER-SEASON HOLDOUT RESULTS


,Dataset,Experiment,Season,BrierScore
0,Men,A_OriginalBaseline,2022,0.245734
1,Men,A_OriginalBaseline,2023,0.222656
2,Men,A_OriginalBaseline,2024,0.203574
3,Men,A_OriginalBaseline,2025,0.168278
4,Men,B_TunedOriginal,2022,0.242093
5,Men,B_TunedOriginal,2023,0.227380
6,Men,B_TunedOriginal,2024,0.208366
7,Men,B_TunedOriginal,2025,0.173720
8,Men,C_NewFeatures,2022,0.240523
9,Men,C_NewFeatures,2023,0.226672



OVERALL HOLDOUT COMPARISON


,Dataset,Experiment,MeanHoldoutBrier
0,Men,A_OriginalBaseline,0.210060
1,Men,B_TunedOriginal,0.212890
2,Men,C_NewFeatures,0.215652
4,Women,B_TunedOriginal,0.145337
5,Women,C_NewFeatures,0.148266
3,Women,A_OriginalBaseline,0.150797


Men | A_OriginalBaseline: actual=0.210060, expected=0.210060
Men | B_TunedOriginal: actual=0.212890, expected=0.212890
Women | A_OriginalBaseline: actual=0.150797, expected=0.150797
Women | B_TunedOriginal: actual=0.145337, expected=0.145337

Previously recorded holdout results reproduced.


In [14]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

# ============================================================
# 1. LOAD EXISTING MASSEY DATASET
# ============================================================

men_massey = pd.read_csv(
    "../data/processed/men_massey.csv"
)

# Massey ordinal ranks must be positive
assert (
    men_massey[
        ["Team1MasseyRank", "Team2MasseyRank"]
    ] > 0
).all().all()

# ============================================================
# 2. CREATE LOG-TRANSFORMED MASSEY FEATURE
# ============================================================

men_massey["LogMasseyRankDiff"] = (
    np.log(men_massey["Team1MasseyRank"])
    - np.log(men_massey["Team2MasseyRank"])
)

assert np.isfinite(
    men_massey["LogMasseyRankDiff"]
).all()

# ============================================================
# 3. DEFINE FEATURE SETS
# ============================================================

base_features = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]

massey_experiments = {
    "M0_Original4": base_features,

    "M1_RawMassey": base_features + [
        "MasseyRankDiff"
    ],

    "M2_LogMassey": base_features + [
        "LogMasseyRankDiff"
    ]
}

# ============================================================
# 4. USE EXISTING MEN'S TUNED PARAMETERS
# ============================================================

men_params = {
    "max_depth": 5,
    "learning_rate": 0.01,
    "n_estimators": 300,
    "subsample": 0.7,
    "reg_lambda": 1,
    "reg_alpha": 0,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss"
}

validation_years = [2017, 2018, 2019, 2021]

# ============================================================
# 5. RUN TEMPORAL CROSS-VALIDATION
# ============================================================

results = []

for experiment_name, features in massey_experiments.items():

    for year in validation_years:

        train = men_massey[
            men_massey["Season"] < year
        ]

        valid = men_massey[
            men_massey["Season"] == year
        ]

        model = XGBClassifier(**men_params)

        model.fit(
            train[features],
            train["Target"]
        )

        predictions = model.predict_proba(
            valid[features]
        )[:, 1]

        brier = brier_score_loss(
            valid["Target"],
            predictions
        )

        results.append({
            "Experiment": experiment_name,
            "ValidationSeason": year,
            "BrierScore": brier
        })

results_df = pd.DataFrame(results)

# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("PER-SEASON BRIER SCORES")

display(
    results_df.pivot(
        index="ValidationSeason",
        columns="Experiment",
        values="BrierScore"
    ).round(6)
)

summary = (
    results_df.groupby("Experiment")["BrierScore"]
    .mean()
    .sort_values()
)

print("\nMEAN CV BRIER")
display(summary.to_frame("MeanCVBrier"))

# ============================================================
# 7. VERIFY PREVIOUSLY RECORDED SCORES
# ============================================================

reference = {
    "M0_Original4": 0.191705,
    "M1_RawMassey": 0.189886
}

for name, expected in reference.items():

    actual = summary[name]

    print(
        f"{name}: actual={actual:.6f}, "
        f"expected={expected:.6f}"
    )

    assert abs(actual - expected) < 0.000002, (
        f"{name} does not match previous results."
    )

print("\nOriginal benchmarks reproduced successfully.")

PER-SEASON BRIER SCORES


Experiment,M0_Original4,M1_RawMassey,M2_LogMassey
ValidationSeason,,,
2017,0.164758,0.169810,0.164622
2018,0.220298,0.210815,0.214595
2019,0.162818,0.155767,0.139790
2021,0.218945,0.223153,0.204665



MEAN CV BRIER


,MeanCVBrier
Experiment,
M2_LogMassey,0.180918
M1_RawMassey,0.189886
M0_Original4,0.191705


M0_Original4: actual=0.191705, expected=0.191705
M1_RawMassey: actual=0.189886, expected=0.189886

Original benchmarks reproduced successfully.
